In [ ]:
# 환경 : 20일 기준
# 행동 : 3가지 +1, 0, -1 (short, cash 관망, long)
# 보상 : (이전 포지션 * 금일 수익률) - 비용(거래변경량 * bps)     # 100bps = 1%
# 포지션을 바꾸면 거래비용이 든다(cost_bps)
# step : 보상 계산 후 시점을 하루 전진, 다음 관측을 반환함
# DQN(replay buffer, target network)

import math, random
from collections import deque
import numpy as np
import pandas as pd
import tensorflow as tf

In [ ]:
# 데이터 읽기 : 수익률 반환
def load_returns(csv_path:str):
  df = pd.read_csv(csv_path)
  close = df['Close'].astype(float).values
  ret = np.zeros_like(close, dtype=np.float32)
  ret[1:] = (close[1:] - close[:-1]) / (close[:-1] + 1e-9)    # 일별 수익률 계산(1e-9 close[:-1]이 0이 되는 것을 방지하고자 아주 작은 값을 더함)
  return ret

# 환경
class TradingEnv:
  def __init__(self, returns:np.ndarray, window=20, cost_bps=10.0):
    assert len(returns) > window + 1, '데이터가 너무 적습니다.'
    self.rets_all = returns.astype(np.float32)
    self.window = window
    self.cost = cost_bps / 10_000.0    # bps : 1/100
    self.reset()

  @property    # method를 변수(필드)처럼 접근 가능
  def obs_dim(self):return self.window + 1    # 관측차원

  @property
  def n_actions(self):return 3

  def reset(self):
    self.t = self.window + 1
    self.pos = 0
    return self._obs()

  def _obs(self):    # 현재 시점의 관측 벡터 생성 ; _ private, __ system
    window = self.rets_all[self.t - self.window:self.t]    # 직전 window의 수익률을 슬라이스
    return np.concatenate([window, [float(self.pos)]]).astype(np.float32)    # 수익률들 + 현재 포지션 결합

  def step(self, action:int):    # 환경 1 스텝 진행(행동 입력 -> 다음 상태/보상/종료)
    new_pos = [-1, 0, +1][action]
    trade_cost = self.cost * abs(new_pos - self.pos)    # 거래 비용(포지션 변화량)
    reward = self.pos * self.rets_all[self.t] - trade_cost    # 보상 계산 : (이전 포지션 * 금일 수익률) - 거래비용
    self.pos = new_pos
    self.t += 1    # 하루 전진
    done = (self.t >= len(self.rets_all) - 1)    # 마지막 전날까지 진행하면 에피소드 종료
    return self._obs(), float(reward), done    # 다음관측, 보상, 종료여부 반환

# Q-Network 신경망 구성
def build_qnet_seq(obs_dim, n_actions):
  model = tf.keras.Sequential([
      tf.keras.layers.Input(shape=(obs_dim,)),
      tf.keras.layers.Dense(64, activation='relu'),
      tf.keras.layers.Dense(64, activation='relu'),
      tf.keras.layers.Dense(n_actions)
  ])
  return model

# 리플레이
class Replay:    # replay buffer 경험 재생 메모리
  def __init__(self, cap=20000):self.buf = deque(maxlen=cap)    # transaction(전이)
  def __len__(self):return len(self.buf)
  def push(self, *tr):self.buf.append(tr)    # 전이(state, action, reward ,state', done) 하나를 버퍼에 추가
  def sample(self, n):
    s = random.sample(self.buf, n)    # 버퍼에서 n개 무작위 추출
    s, a, r, ns, d = zip(*s)    # 전이 tuple을 각 배열로 분리
    return (np.array(s, np.float32),
            np.array(a, np.float32),
            np.array(r, np.float32),
            np.array(ns, np.float32),
            np.array(d, np.float32))

# DQN Agent(기본형)
class DQN:
  def __init__(self, obs_dim, n_actions, lr=3e-4, gammer=0.99, batch=32):
    self.q = build_qnet_seq(obs_dim, n_actions)    # 메인 네트워크
    self.tgt = build_qnet_seq(obs_dim, n_actions)    # 타겟 네트워크
    self.tgt.set_weights(self.q.get_weights())
    self.opt = tf.keras.optimizers.Adam(lr)
    self.gammer, self.batch = gammer, batch

    self.buf = Replay()
    self.loss_fn = tf.keras.losses.Huber()    # Huber loss : MSE보다 이상치에 덜 민감
    self.eps = 1.0
    self.eps_min = 0.05
    self.eps_decay = 0.995
    self.n_actions = n_actions

  def act(self, obs):
    if random.random() < self.eps:
      return random.randrange(self.n_actions)
    qv = self.q(obs[None, :], training=False).numpy()[0]    # q(state, ) 계산
    return int(np.argmax(qv))    # q 값이 최대인 행동(활용)


  def update(self):    # 파라미터 갱신
    if len(self.buf) < self.batch:return
    s, a, r, ns, d = self.buf.sample(self.batch)    # 미니배치 전이 샘플링

    a_oh = tf.one_hot(a, self.n_actions)    # 행동 인덱스를 원핫 벡터로 변환 (Q(s, )에서 Q(s, a) 추출용)
    with tf.GradientTape() as tape:
      q_sa = tf.reduce_sum(self.q(s) * a_oh, axis=1)    # 원핫의 내적을 계산
      q_next = tf.reduce_max(self.tgt(ns), axis=1)    # 타겟 네트워크로 다음 상태의 최대 Q
      y = r + (1 - d) * self.gammer * q_next    # 벨만 방정식
      loss = self.loss_fn(y, q_sa)    # 손실 = Huber(y, Q(s, a))

    q = tape.gradient(loss, self.q.trainable_variables)    # 온라인 네트워크에 대한 그래디언트 계산
    self.opt.apply_gradients(zip(q, self.q.trainable_variables))    # 경사 하강 스텝 적용
    self.tgt.set_weights(self.q.get_weights())
    self.eps = max(self.eps * self.eps_decay, self.eps_min)    # 탐험 비율을 점진적으로 줄임

def train(csv_path='/content/prices.csv', window=20, cost_bps=10.0, episodes=5):
  rets = load_returns(csv_path)
  # print(rets)
  env = TradingEnv(rets, window, cost_bps)
  agent = DQN(env.obs_dim, env.n_actions)

  equity = []    # 누적 pnl 추적 리스트(성과 지표용)
  for ep in range(1, episodes + 1):
    obs = env.reset()    # 환경 초기화
    done, ep_pnl = False, 0.0    # done, 누적 pnl 초기화
    while not done:    # 데이터 끝까지 하루씩 진행
      act = agent.act(obs)    # 행동 선택
      nobs, r, done = env.step(act)    # 다음 관측값, 보상, 종료를 반환받음
      agent.buf.push(obs, act, r, nobs, float(done))    # 전이(s, a, r, s', done을 버퍼에 저장)
      agent.update()
      obs = nobs
      ep_pnl += r
      equity.append(ep_pnl)
    print(f"[EP {ep}/{episodes}] PnL={ep_pnl:.4f}, eps={agent.eps:.3f}")

  # 간단한 모델 요약
  equity = np.array(equity)
  daily_ret = np.diff(equity, prepend=0)    # 일별 PnL 증분(=보상 시퀀스) 추정
  sharp = daily_ret.mean() / (daily_ret.std() + 1e-9) * np.sqrt(252)    # 샤프 비율 근사 : (일별 수익률 평균 / 일별 수익률 표준편차) * root252
  print("요약 결과")
  print(f"Final PnL : {equity[-1]:.6f}")
  print(f"sharp(위험대비 수익 척도) : {sharp:.3f}")

  agent.q.save('dqn_model.keras')    # 모델 저장

if __name__ == "__main__":
  train(csv_path='/content/prices.csv', window=20, cost_bps=10.0, episodes=100)